# problem statement

##### spam filterng using naive bayes classfier in order to predict whether new mailbased on its content,can be categories spam and not spam 

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import string

# Load Data

In [2]:
data= pd.read_csv("spam.csv")

In [3]:
data

,Class,Message,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [4]:
data.drop(["Unnamed: 2","Unnamed: 3","Unnamed: 4"],axis=1,inplace=True)

In [5]:
data

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
# check first 8 messages
data.head(8)


,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...


In [7]:
#check last 8 messages
data.tail(8)

,Class,Message
5564,ham,Why don't you wait 'til at least wednesday to ...
5565,ham,Huh y lei...
5566,spam,REMINDER FROM O2: To get 2.50 pounds free call...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [8]:
#check first message
data.loc[:0]

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."


In [9]:
# check summary of data
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Class    5572 non-null   object
 1   Message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [10]:
# create a column of message length
data["length"]=data["Message"].apply(len)
data

,Class,Message,length
0,ham,"Go until jurong point, crazy.. Available only ...",111
1,ham,Ok lar... Joking wif u oni...,29
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
3,ham,U dun say so early hor... U c already then say...,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",61
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,161
5568,ham,Will Ì_ b going to esplanade fr home?,37
5569,ham,"Pity, * was in mood for that. So...any other s...",57
5570,ham,The guy did some bitching but I acted like i'd...,125


In [11]:
# The mail has 2 categories i.e. ham,spam 
# lets check the count of each class
data.groupby("Class").count()

,Message,length
Class,,
ham,4825,4825
spam,747,747


# Data Visualization

In [12]:
data['length'].describe()

count    5572.000000
mean       80.118808
std        59.690841
min         2.000000
25%        36.000000
50%        61.000000
75%       121.000000
max       910.000000
Name: length, dtype: float64

In [13]:
# see the max length of message is shown as 910 char so we just check this message 
data[data["length"]==910]["Message"]

1084    For me the love should start with attraction.i...
Name: Message, dtype: object

In [14]:
data[data["length"]==910]["Message"].iloc[0]

"For me the love should start with attraction.i should feel that I need her every time around me.she should be the first thing which comes in my thoughts.I would start the day and end it with her.she should be there every time I dream.love will be then when my every breath has her name.my life should happen around her.my life will be named to her.I would cry for her.will give all my happiness and take all her sorrows.I will be ready to fight with anyone for her.I will be in love when I will be doing the craziest things for her.love will be when I don't have to proove anyone that my girl is the most beautiful lady on the whole planet.I will always be singing praises for her.love will be when I start up making chicken curry and end up makiing sambar.life will be the most beautiful then.will get every morning and thank god for the day because she is with me.I would like to say a lot..will tell later.."

In [15]:
# see the min length of meesage is shown as 2 char so we just check this message 
data[data["length"]==2]["Message"].iloc[0]

'Ok'

In [16]:
#Text pre processing 
#Assign objects to the target variable
dObject=data["Class"].values
dObject

array(['ham', 'ham', 'spam', ..., 'ham', 'ham', 'ham'], dtype=object)

In [17]:
# Assign ham as 1
data.loc[data["Class"]=="ham","Class"]=1

In [18]:
# Assign spam as 0
data.loc[data["Class"]=="spam","Class"]=0

In [19]:
# Check object of target variable
Dobject2=data["Class"].values
Dobject2

array([1, 1, 0, ..., 1, 1, 1], dtype=object)

In [20]:
# Remove Punctuation
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [21]:
# Why it is important
"This Message is spam"=="This i message is spam."

False

In [22]:
# Lets remove Punctuations
def remove_punc(text):
    text = "".join([char for char in text if char not in string.punctuation])
    return text


data['text_clean']=data["Message"].apply (lambda x : remove_punc(x))    


In [23]:
data

,Class,Message,length,text_clean
0,1,"Go until jurong point, crazy.. Available only ...",111,Go until jurong point crazy Available only in ...
1,1,Ok lar... Joking wif u oni...,29,Ok lar Joking wif u oni
2,0,Free entry in 2 a wkly comp to win FA Cup fina...,155,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...,49,U dun say so early hor U c already then say
4,1,"Nah I don't think he goes to usf, he lives aro...",61,Nah I dont think he goes to usf he lives aroun...
...,...,...,...,...
5567,0,This is the 2nd time we have tried 2 contact u...,161,This is the 2nd time we have tried 2 contact u...
5568,1,Will Ì_ b going to esplanade fr home?,37,Will Ì b going to esplanade fr home
5569,1,"Pity, * was in mood for that. So...any other s...",57,Pity was in mood for that Soany other suggest...
5570,1,The guy did some bitching but I acted like i'd...,125,The guy did some bitching but I acted like id ...


In [24]:
# Convert categorical data into numerical data countvectorizer
cv= CountVectorizer(stop_words="english")
cv

CountVectorizer(stop_words='english')

In [25]:
#split x and y
xSet=data["text_clean"].values
ySet=data["Class"].values
xSet

array(['Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat',
       'Ok lar Joking wif u oni',
       'Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005 Text FA to 87121 to receive entry questionstd txt rateTCs apply 08452810075over18s',
       ..., 'Pity  was in mood for that Soany other suggestions',
       'The guy did some bitching but I acted like id be interested in buying something else next week and he gave it to us for free',
       'Rofl Its true to its name'], dtype=object)

In [26]:
ySet=ySet.astype('int')
ySet

array([1, 1, 0, ..., 1, 1, 1])

In [27]:
xSet

array(['Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat',
       'Ok lar Joking wif u oni',
       'Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005 Text FA to 87121 to receive entry questionstd txt rateTCs apply 08452810075over18s',
       ..., 'Pity  was in mood for that Soany other suggestions',
       'The guy did some bitching but I acted like id be interested in buying something else next week and he gave it to us for free',
       'Rofl Its true to its name'], dtype=object)

In [28]:
# Spliting Train an dTest Data
xSet_train,xSet_test,ySet_train,ySet_test=train_test_split(xSet,ySet,test_size=0.2,random_state=3)

In [29]:
xSet_train_CV=cv.fit_transform(xSet_train)
xSet_train_CV

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 34308 stored elements and shape (4457, 8167)>

# Training Model

In [30]:
# With message repreented as vector,we can finally train our spam / hsm classifier.
# WE actially use multiple type of classifier algorithm for a veriety of reson use can use naive bayes algorithms.

In [31]:
# initialising model
NB=MultinomialNB()
NB

MultinomialNB()

In [32]:
# feed data to the model
NB.fit(xSet_train_CV,ySet_train)

MultinomialNB()

In [38]:
# let's test CV on our test data
xSet_test_CV=cv.transform(xSet_test)

In [39]:
# predict for xSet_test_CV
ySet_predict=NB.predict(xSet_test_CV)
ySet_predict

array([1, 0, 1, ..., 1, 1, 1])

In [41]:
accuracy_score=accuracy_score(ySet_test,ySet_predict)*100
print("prediction accuracy is",accuracy_score)

prediction accuracy is 98.38565022421525


In [45]:
# Spam Classification Application
msg=input("enter your message")
msginput=cv.transform([msg])
predict=NB.predict(msginput)
if (predict[0]==0):
    print("------check spam folder")
else:
    print("-----check inbox")


enter your message message


-----check inbox


In [46]:
# spiliting x and y
x=data["text_clean"].values
y=data["Class"].values
y

array([1, 1, 0, ..., 1, 1, 1], dtype=object)

In [47]:
#data y is in numerical but represent object as a type so convert into int
y=y.astype('int')
y

array([1, 1, 0, ..., 1, 1, 1])

In [48]:
type(x)

numpy.ndarray

In [49]:
# text preprocessing and feature vectorisation
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer()
X=tf.fit_transform(x)

In [50]:
X.shape

(5572, 9489)

In [51]:
#print feature name of tfidf vectoriser
tf.get_feature_names_out()

array(['008704050406', '0089my', '0121', ..., 'ûïharry', 'ûò', 'ûówell'],
      dtype=object)

In [52]:
len(tf.get_feature_names_out())

9489

In [53]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 72459 stored elements and shape (5572, 9489)>

In [54]:
X=X.toarray()

In [55]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [56]:
#creating training and testing
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,random_state=6)

In [57]:
#model creation
from sklearn.naive_bayes import BernoulliNB

In [58]:
#model initilising
nb=BernoulliNB()

# fit the model
nb.fit(X_train,y_train)

#get the prediction
y_hat=nb.predict(X_test)

In [59]:
y_hat

array([1, 1, 1, ..., 1, 1, 1])

In [63]:
# evaluating a model
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

In [65]:
print("classification Report is",classification_report(y_test,y_hat))

classification Report is               precision    recall  f1-score   support

           0       0.99      0.85      0.91       201
           1       0.98      1.00      0.99      1192

    accuracy                           0.98      1393
   macro avg       0.98      0.92      0.95      1393
weighted avg       0.98      0.98      0.98      1393



In [66]:
#confusion metrics
pd.crosstab(y_test,y_hat)

col_0,0,1
row_0,,
0,171,30
1,2,1190


In [67]:
accuracy_score=accuracy_score(y_test,y_hat)*100
print("accuracy score is",accuracy_score)

accuracy score is 97.70279971284997
